# Imvo project: all GPU work on Colab

**Use this in the browser** (colab.research.google.com): downloads do not work from the VS Code Colab extension.

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Cell 2 asks for `colab/colab_bundle.zip` (make it with `python src/make_colab_bundle.py`).
3. The last cell downloads `embeddings_all.zip` (and `results_all.zip` if you ran the analysis cell). Unzip both into the project folder.

Total GPU time on a T4: ~40 min for LaBSE (4 spelling variants) + ~15 min for the second encoder.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from google.colab import files
up = files.upload()  # choose colab_bundle.zip
!unzip -o -q colab_bundle.zip && ls src data/processed
!ls labels 2>/dev/null || echo "no labels bundled (provisional threshold will be used)"

Saving colab_bundle.zip to colab_bundle.zip
data/processed:
chunks.parquet	normaliser_candidates.parquet

src:
build_lexicon.py  embed.py		prepare_data.py  synthetic_check.py
classify.py	  make_colab_bundle.py	qualitative.py	 tool_metrics.py
common.py	  normalise.py		run_all.py
compare.py	  pairs.py		smoke_test.py
no labels bundled (provisional threshold will be used)


In [3]:
!pip -q install sentence-transformers pyarrow tabulate
# regenerate the modernised variants from the shipped candidate table (deterministic)
!for r in all asp prefix; do python src/normalise.py apply --rules $r; done

modern: 9.1% of isiXhosa tokens changed ({'asp': 289816, 'prefix_common': 797, 'prefix_proper': 47517}); 96.3% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_asp: 7.8% of isiXhosa tokens changed ({'asp': 289816}); 95.0% of chunks; thresholds 2,5 [provisional (no labelled words yet)]
modern_prefix: 1.3% of isiXhosa tokens changed ({'prefix_common': 797, 'prefix_proper': 47517}); 44.5% of chunks; thresholds 2,5 [provisional (no labelled words yet)]


## LaBSE: raw + three spelling variants
Modern variants reuse raw embeddings for unchanged chunks (all English text).

In [4]:
!python src/embed.py --variant raw --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_asp --chunks data/processed/chunks_modern_asp.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_prefix --chunks data/processed/chunks_modern_prefix.parquet --reuse raw --batch 128 --shard 8192

modules.json: 100% 461/461 [00:00<00:00, 2.45MB/s]
config_sentence_transformers.json: 100% 122/122 [00:00<00:00, 758kB/s]
README.md: 100% 2.02k/2.02k [00:00<00:00, 7.07MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 233kB/s]
config.json: 100% 804/804 [00:00<00:00, 5.06MB/s]

model.safetensors: downloading bytes:   9% 170M/1.88G [00:01<00:07, 217MB/s, 12.0MB/s  ]
model.safetensors: downloading bytes:  47% 894M/1.88G [00:03<00:02, 429MB/s, 73.4MB/s  ]
model.safetensors: downloading bytes:  58% 1.08G/1.88G [00:04<00:03, 221MB/s, 87.1MB/s  ]
model.safetensors: downloading bytes:  67% 1.27G/1.88G [00:12<00:09, 66.1MB/s, 55.5MB/s  ]
model.safetensors: downloading bytes:  95% 1.79G/1.88G [00:13<00:00, 352MB/s, 90.8MB/s  ]
model.safetensors: reconstructing file:  71% 1.34G/1.88G [00:13<00:05, 108MB/s, 82.0MB/s  ] 
model.safetensors: downloading bytes: 100% 1.79G/1.79G [00:15<00:00, 113MB/s, 90.7MB/s  ]
model.safetensors: reconstructing file: 100% 1.88G/1.88G [00:15<00:00, 119MB/s

## Second encoder (robustness): multilingual-e5-base, raw + modern

In [5]:
E5 = 'intfloat/multilingual-e5-base'
!python src/embed.py --variant raw --model {E5} --prefix 'query: ' --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --model {E5} --prefix 'query: ' --reuse raw --batch 128 --shard 8192

modules.json: 100% 387/387 [00:00<00:00, 1.51MB/s]
README.md: 100% 179k/179k [00:00<00:00, 149MB/s]
sentence_bert_config.json: 100% 57.0/57.0 [00:00<00:00, 375kB/s]
config.json: 100% 694/694 [00:00<00:00, 4.47MB/s]

model.safetensors: downloading bytes:  24% 262M/1.11G [00:01<00:04, 202MB/s, 23.1MB/s  ]
model.safetensors: downloading bytes:  26% 295M/1.11G [00:01<00:04, 187MB/s, 25.7MB/s  ]
model.safetensors: downloading bytes:  44% 492M/1.11G [00:02<00:02, 266MB/s, 40.6MB/s  ]
model.safetensors: downloading bytes:  51% 565M/1.11G [00:02<00:02, 254MB/s, 46.3MB/s  ]
model.safetensors: downloading bytes:  56% 624M/1.11G [00:03<00:02, 186MB/s, 51.0MB/s  ]
model.safetensors: downloading bytes:  63% 702M/1.11G [00:03<00:02, 141MB/s, 55.6MB/s  ]
model.safetensors: downloading bytes:  67% 741M/1.11G [00:04<00:02, 157MB/s, 57.3MB/s  ]
model.safetensors: reconstructing file:  78% 872M/1.11G [00:06<00:01, 121MB/s, 64.6MB/s  ]
model.safetensors: downloading bytes: 100% 741M/741M [00:08<00:00, 92.

In [6]:
import numpy as np, glob
for f in sorted(glob.glob('data/embeddings/*.npy')):
    e = np.load(f); n = np.linalg.norm(e.astype('float32'), axis=1)
    print(f, e.shape, 'norms', n.min().round(3), n.max().round(3))

data/embeddings/modern_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_asp_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0
data/embeddings/modern_prefix_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_LaBSE.npy (68392, 768) norms 1.0 1.0
data/embeddings/raw_multilingual-e5-base.npy (68392, 768) norms 1.0 1.0


## Optional: run the analysis here too
Labels are optional: if `labels/pairs_labelled.csv` is in the bundle it is used; otherwise the provisional `THRESHOLD` below is used for LaBSE. E5 similarities sit in a different range, so the E5 run uses the threshold that keeps the same share of its candidates as LaBSE keeps at `THRESHOLD` (`pairs.py match`); run the LaBSE line first. The LaBSE run also repeats the raw classifier with stricter pairs (0.65) into `results/raw_thr0.65`. The E5 run analyses raw + modern only (the variants embedded above).

In [7]:
THRESHOLD = 0.55   # provisional; ignored if labels/pairs_labelled.csv was bundled
!python src/run_all.py analyse --threshold {THRESHOLD}
!python src/run_all.py analyse --model-tag multilingual-e5-base --threshold {THRESHOLD}


$ python src/pairs.py candidates --variant raw --model-tag LaBSE
8,066 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1243.0  0.509  0.080  0.328  0.451  0.493  0.550  0.792
train  5618.0  0.509  0.079  0.272  0.454  0.494  0.548  0.875
val    1205.0  0.511  0.077  0.297  0.458  0.492  0.549  0.821

$ python src/pairs.py candidates --variant modern --model-tag LaBSE
8,197 mutual-NN candidates from 511 issues (splits embedded: ['test', 'train', 'val'])
        count   mean    std    min    25%    50%    75%    max
split                                                         
test   1239.0  0.515  0.083  0.334  0.455  0.497  0.560  0.816
train  5717.0  0.513  0.081  0.272  0.456  0.497  0.553  0.875
val    1241.0  0.515  0.080  0.339  0.460  0.497  0.558  0.831

$ python src/pairs.py candidates --variant modern_asp --mode

In [8]:
!cd data/embeddings && zip -q -0 ../../embeddings_all.zip *.npy *_ids.txt
!zip -q -r results_all.zip results 2>/dev/null; ls -la *.zip
files.download('embeddings_all.zip')
import os
if os.path.exists('results_all.zip'): files.download('results_all.zip')

-rw-r--r-- 1 root root  27093668 Oct 21 09:10 colab_bundle.zip
-rw-r--r-- 1 root root 636595192 Oct 21 10:07 embeddings_all.zip
-rw-r--r-- 1 root root  21263894 Oct 21 10:07 results_all.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>